In [ ]:
import os
import sys

sys.path.insert(0, os.path.abspath(".."))

import dotenv
from neo4j import GraphDatabase
from neo4j_graphrag.embeddings.openai import OpenAIEmbeddings
from neo4j_graphrag.generation import RagTemplate
from neo4j_graphrag.generation.graphrag import GraphRAG
from neo4j_graphrag.llm.openai_llm import OpenAILLM
from neo4j_graphrag.retrievers import HybridRetriever
from core import chat

from core.models import Settings

dotenv.load_dotenv()

VECTOR_INDEX = "coins_embeddings_index"
FULL_TEXT_INDEX = "coins_text_index"
settings = Settings()


# query_text = "What are the Roman coins where an emperor standing frontally, dressed militarily, and facing right on the reverse?"
query_text = "What is the coin with DN HONORIVS PF AVG lettering?"
# query_text = "who is Honorius?"


class RagChatService(chat.ChatService):
    def __init__(self):
        driver = GraphDatabase.driver(
            settings.neo4j_uri, auth=(settings.neo4j_username, settings.neo4j_password)
        )
        embedder = OpenAIEmbeddings(
            model=settings.embedding_model, api_key=settings.openai_api_key
        )
        retriever = HybridRetriever(
            driver,
            VECTOR_INDEX,
            FULL_TEXT_INDEX,
            embedder,
            neo4j_database=settings.neo4j_database,
        )
        llm = OpenAILLM(
            settings.chat_model,
            model_params={
                "temperature": settings.temperature,
                "top_p": settings.top_p,
            },
            api_key=settings.openai_api_key,
        )
        prompt_template = RagTemplate(
            template="""
Context (Roman coin data in Python dictionary format):
{context}

Question:
{query_text}

Answer:
""",
            expected_inputs=["context", "query_text"],
            system_instructions="""
You are a knowledgeable historian specializing in Roman coinage. 

IMPORTANT: The context contains Roman coin data formatted as Python dictionary strings. Each entry looks like:
{{'title': 'RIC IX Rome 20C', 'record_id': 'ric.9.rom.20C', 'from_year': 367, 'to_year': 375, 'object_type': 'Coin', 'description': 'Obverse: ... Reverse: ...'}}

Your task:
1. Read and parse each dictionary entry in the context
2. Look at the 'description' field which contains two parts separated by newlines:
   - "Obverse:" describes the front of the coin
   - "Reverse:" describes the back of the coin
3. Answer the user's question based on these descriptions

When the user asks about coins:
- FIRST, check if any coins in the provided context match their criteria
- If matching coins are found in the context:
  * List ALL matching coins from the context
  * For each coin, provide:
    - The coin title (from 'title' field)
    - The date range (from 'from_year' and 'to_year' fields)
    - The relevant description (from 'description' field)
  * Use plain, conversational language

- If NO coins in the context match the criteria OR if the context is empty:
  * Draw upon your knowledge of Roman history and numismatics
  * Provide a general answer about such coins if they exist in Roman coinage
  * Mention specific examples from your knowledge (e.g., coin types, emperors, common reverse designs)
  * Note that you're providing information from general historical knowledge since the database didn't contain matching examples

Example response format when coins are found in context:
"Based on the provided coin data, I found the following Roman coins with an emperor standing on the reverse:

1. RIC IX Rome 20C (367-375 CE): This coin shows an emperor standing facing with his head turned right, holding a labarum and Victory on globe.

2. RIC VII Rome 45 (310-317 CE): This coin depicts an emperor in military dress standing frontally..."

Example response format when using general knowledge:
"While the database doesn't contain specific examples matching your criteria, Roman coinage frequently featured emperors standing on the reverse. Common examples include coins of Constantine I and his successors, which often depicted the emperor in military dress standing frontally. These reverse types typically showed the emperor holding military standards (labarum or vexillum) and victory symbols..."

DO NOT use markdown formatting. Use plain text only.
DO NOT say there's not enough information if matching coins exist in the context.
ALWAYS provide a helpful answer, either from the context or from your historical knowledge.
""",
        )

        self._graph_rag = GraphRAG(llm=llm, retriever=retriever, prompt_template=prompt_template)

    def chat(self, query: str, history: list[tuple[str, str]] | None = None) -> str:
        response = self._graph_rag.search(
            query, retriever_config={"top_k": settings.top_k}, return_context=True
        )
        return response.answer
    

chat.run(RagChatService())